# Favorite at 90c Analysis

Simple question: in each 15-minute contract, does the favorite reach 90c?
If so, does it win at settlement?

Strategy: buy the favorite at 90c across all 11 markets.
- Profit per win: (100 - 90) × 10 lots = $1.00
- Loss per loss: 90 × 10 lots = $9.00 (before stop-loss)
- Goal: $1 × 11 markets × 4 cycles/hr = $44/hr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import time

TICK_CSV = "/content/drive/MyDrive/commod15min_ticks.csv"
OUT_CSV  = "/content/drive/MyDrive/fav90_analysis.csv"

SETTLE_WINDOW_SEC = 30.0
LOTS = 10

# windows to test: last N seconds
WINDOWS = [420, 360, 300, 240, 180, 120, 90, 60, 50]

# favorite price thresholds to check
THRESHOLDS = [85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]

print(f"Windows: {WINDOWS}")
print(f"Thresholds: {THRESHOLDS}")

In [ ]:
t0 = time.time()
print(f"loading {TICK_CSV} ...")
df = pd.read_csv(TICK_CSV, usecols=[
    "ticker", "asset", "yes_mid", "sec_to_close",
])
df = df.dropna(subset=["yes_mid", "sec_to_close"])
df["minute_in_cycle"] = 15.0 - df["sec_to_close"] / 60.0
df = df[(df["minute_in_cycle"] >= 0) & (df["minute_in_cycle"] <= 15)]

def determine_settlement(minute, ymid):
    sec_to_close = (15.0 - minute) * 60.0
    mask = sec_to_close <= SETTLE_WINDOW_SEC
    if mask.any():
        return ymid[mask].mean() > 50.0
    return ymid[-1] > 50.0

windows = []
for ticker, g in df.groupby("ticker", sort=False):
    g = g.sort_values("minute_in_cycle")
    minute = g["minute_in_cycle"].to_numpy()
    ymid   = g["yes_mid"].to_numpy(dtype=float)
    stc    = g["sec_to_close"].to_numpy(dtype=float)
    asset  = g["asset"].iloc[0]
    yes_won = determine_settlement(minute, ymid)
    windows.append((asset, minute, ymid, stc, yes_won))

print(f"loaded {len(windows)} contracts in {time.time()-t0:.1f}s")

In [ ]:
rows = []

for asset_name in sorted(set(a for a, _, _, _, _ in windows)):
    asset_windows = [(m, y, s, w) for a, m, y, s, w in windows if a == asset_name]
    n_contracts = len(asset_windows)

    for win_sec in WINDOWS:
        for thresh in THRESHOLDS:
            contracts_with_hit = 0
            fav_wins = 0
            fav_losses = 0
            entry_prices = []    # actual price when first hitting threshold
            
            for minute, ymid, stc, yes_won in asset_windows:
                # ticks in the observation window
                mask = stc <= win_sec
                if not mask.any():
                    continue
                
                w_ymid = ymid[mask]
                fav_price = np.maximum(w_ymid, 100.0 - w_ymid)
                fav_is_yes = w_ymid >= 50.0
                
                # does favorite reach the threshold?
                hits = fav_price >= thresh
                if not hits.any():
                    continue
                
                contracts_with_hit += 1
                
                # entry at first tick where favorite >= threshold
                fi = int(np.argmax(hits))
                px = float(fav_price[fi])
                iy = bool(fav_is_yes[fi])
                won = (iy and yes_won) or (not iy and not yes_won)
                
                entry_prices.append(px)
                if won:
                    fav_wins += 1
                else:
                    fav_losses += 1
            
            if contracts_with_hit == 0:
                continue
            
            avg_entry = np.mean(entry_prices)
            profit_per_win = (100.0 - avg_entry) * LOTS
            loss_per_loss = avg_entry * LOTS
            total_profit_wins = sum((100.0 - p) * LOTS for p in entry_prices[:contracts_with_hit] 
                                     if True)  # need per-trade
            # recalc properly
            win_profit = sum((100.0 - p) * LOTS 
                            for p, w in zip(entry_prices, 
                                [True]*fav_wins + [False]*fav_losses) if w)
            loss_cost = sum(p * LOTS 
                           for p, w in zip(entry_prices,
                               [True]*fav_wins + [False]*fav_losses) if not w)
            
            rows.append({
                'asset': asset_name,
                'window_sec': win_sec,
                'threshold_c': thresh,
                'total_contracts': n_contracts,
                'contracts_with_hit': contracts_with_hit,
                'hit_rate': round(contracts_with_hit / n_contracts, 4),
                'fav_wins': fav_wins,
                'fav_losses': fav_losses,
                'win_rate': round(fav_wins / contracts_with_hit, 4),
                'avg_entry_price_c': round(avg_entry, 2),
                'profit_per_win_c': round((100.0 - avg_entry) * LOTS, 1),
                'loss_per_loss_c': round(avg_entry * LOTS, 1),
            })

result = pd.DataFrame(rows)
result.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV} ({len(result)} rows)")
print(f"done in {time.time()-t0:.1f}s")

In [ ]:
# Focus: favorite >= 90c, last 7 minutes (420s)
print("="*100)
print("FAVORITE >= 90c  |  LAST 7 MINUTES (420s)")
print("Profit per win = (100 - entry) × 10 lots | Loss per loss = entry × 10 lots")
print("="*100)

focus = result[(result['threshold_c'] == 90) & (result['window_sec'] == 420)].copy()
focus = focus.sort_values('asset')

cols = ['asset', 'total_contracts', 'contracts_with_hit', 'hit_rate',
        'fav_wins', 'fav_losses', 'win_rate', 'avg_entry_price_c',
        'profit_per_win_c', 'loss_per_loss_c']
print(focus[cols].to_string(index=False))

print(f"\n  TOTALS across all markets:")
print(f"  Contracts with 90c+ favorite: {focus['contracts_with_hit'].sum()}")
print(f"  Wins: {focus['fav_wins'].sum()}  Losses: {focus['fav_losses'].sum()}")
total_hit = focus['contracts_with_hit'].sum()
total_win = focus['fav_wins'].sum()
print(f"  Overall win rate: {total_win/total_hit:.1%}")
print(f"  Avg entry price: {focus['avg_entry_price_c'].mean():.1f}c")
profit_est = total_win * (100 - focus['avg_entry_price_c'].mean()) * LOTS / 100
loss_est = focus['fav_losses'].sum() * focus['avg_entry_price_c'].mean() * LOTS / 100
print(f"  Est. total win profit: ${profit_est:,.2f}")
print(f"  Est. total loss cost:  ${loss_est:,.2f}")
print(f"  Net (no stop-loss):    ${profit_est - loss_est:,.2f}")

In [ ]:
# How threshold affects win rate (last 7 minutes, all assets combined)
print("="*100)
print("WIN RATE BY THRESHOLD  |  LAST 7 MINUTES (420s)  |  ALL ASSETS")
print("="*100)

for thresh in THRESHOLDS:
    sub = result[(result['threshold_c'] == thresh) & (result['window_sec'] == 420)]
    hits = sub['contracts_with_hit'].sum()
    wins = sub['fav_wins'].sum()
    losses = sub['fav_losses'].sum()
    contracts = sub['total_contracts'].sum()
    avg_px = sub['avg_entry_price_c'].mean()
    pnl_win = (100 - avg_px) * LOTS
    pnl_loss = avg_px * LOTS
    net = wins * pnl_win - losses * pnl_loss
    print(f"  >= {thresh}c: {hits:5d} hits ({hits/contracts*100:5.1f}% of contracts) | "
          f"wins {wins:4d}  losses {losses:3d}  WR {wins/hits*100:5.1f}% | "
          f"avg entry {avg_px:.1f}c | "
          f"net ${net/100:+,.2f}")

In [ ]:
# How window size affects 90c results
print("="*100)
print("90c FAVORITE BY WINDOW SIZE  |  ALL ASSETS")
print("="*100)

for win_sec in WINDOWS:
    sub = result[(result['threshold_c'] == 90) & (result['window_sec'] == win_sec)]
    hits = sub['contracts_with_hit'].sum()
    wins = sub['fav_wins'].sum()
    losses = sub['fav_losses'].sum()
    contracts = sub['total_contracts'].sum()
    if hits == 0:
        continue
    avg_px = sub['avg_entry_price_c'].mean()
    pnl_win = (100 - avg_px) * LOTS
    pnl_loss = avg_px * LOTS
    net = wins * pnl_win - losses * pnl_loss
    per_mkt_per_cycle = hits / 11  # avg per market
    print(f"  last {win_sec:3d}s ({win_sec/60:4.1f}m): {hits:5d} hits | "
          f"wins {wins:4d}  losses {losses:3d}  WR {wins/hits*100:5.1f}% | "
          f"avg/mkt/cycle: {per_mkt_per_cycle:.1f} | "
          f"net ${net/100:+,.2f}")

In [ ]:
# Per-market reliability: how often does EACH market have a 90c favorite
# in each 15-minute cycle? (last 7 min window)
print("="*100)
print("PER-MARKET RELIABILITY: 90c favorite in last 7 minutes")
print("How many of the 15-min cycles have a 90c+ favorite available?")
print("="*100)

focus = result[(result['threshold_c'] == 90) & (result['window_sec'] == 420)].sort_values('asset')
for _, row in focus.iterrows():
    bar = '█' * int(row['hit_rate'] * 50)
    print(f"  {row['asset']:>10s}: {row['contracts_with_hit']:4d}/{row['total_contracts']:4d} "
          f"({row['hit_rate']*100:5.1f}%) {bar}")
    print(f"             wins {row['fav_wins']:3d}  losses {row['fav_losses']:3d}  "
          f"WR {row['win_rate']*100:.1f}%  "
          f"avg entry {row['avg_entry_price_c']:.1f}c")

print(f"\n  For $44/hr you need ALL 11 markets hitting 90c in every cycle.")
print(f"  The limiting factor is the market with the lowest hit rate.")
min_hit = focus.loc[focus['hit_rate'].idxmin()]
print(f"  Lowest: {min_hit['asset']} at {min_hit['hit_rate']*100:.1f}%")
max_hit = focus.loc[focus['hit_rate'].idxmax()]
print(f"  Highest: {max_hit['asset']} at {max_hit['hit_rate']*100:.1f}%")

In [ ]:
# Hourly earnings estimate
print("="*100)
print("HOURLY EARNINGS ESTIMATE (4 cycles/hr × 11 markets)")
print("="*100)

for thresh in [88, 89, 90, 91, 92, 93, 94, 95]:
    sub = result[(result['threshold_c'] == thresh) & (result['window_sec'] == 420)]
    if sub.empty:
        continue
    
    # per market per cycle: probability of having a trade
    # and probability of winning
    trades_per_hr = 0
    wins_per_hr = 0
    losses_per_hr = 0
    avg_px = sub['avg_entry_price_c'].mean()
    
    for _, row in sub.iterrows():
        mkt_trades_per_cycle = row['hit_rate']  # prob of trade per cycle
        mkt_wins_per_cycle = row['hit_rate'] * row['win_rate']
        mkt_losses_per_cycle = row['hit_rate'] * (1 - row['win_rate'])
        trades_per_hr += mkt_trades_per_cycle * 4
        wins_per_hr += mkt_wins_per_cycle * 4
        losses_per_hr += mkt_losses_per_cycle * 4
    
    win_profit = wins_per_hr * (100 - avg_px) * LOTS / 100
    loss_cost = losses_per_hr * avg_px * LOTS / 100
    net_per_hr = win_profit - loss_cost
    
    print(f"  >= {thresh}c: {trades_per_hr:5.1f} trades/hr  "
          f"wins {wins_per_hr:5.1f}  losses {losses_per_hr:4.1f}  "
          f"win$ ${win_profit:6.2f}  loss$ ${loss_cost:6.2f}  "
          f"NET ${net_per_hr:+7.2f}/hr")

print(f"\n  Note: loss column is BEFORE stop-loss strategy.")
print(f"  Your stop-loss/reversal strategy (phase 2) would reduce these losses.")